# G1 step-length training (Colab)
Generated by `scripts/make_colab.py`. Edit `g1pipe/` in the repo, not this notebook.

1. Runtime → Change runtime type → **T4 GPU**
2. Set `TIMESTEPS` / `RUN_NAME` below
3. Run all. Checkpoints are written to Google Drive every eval, so a disconnect loses at most one eval interval.

In [ ]:
TIMESTEPS = 150_000_000   # 5_000_000 for a quick probe
RUN_NAME = 'g1-steplength-v1'
EXTRA_ARGS = []            # e.g. ['--no-dr'] for experiment E3
USE_DRIVE = True

In [ ]:
!nvidia-smi
!pip install -q 'playground==0.2.0' 'brax==0.14.2' 'jax[cuda12]==0.7.2' 'flax==0.12.0' 'mujoco==3.14.0' 'mujoco-mjx==3.14.0' 'warp-lang==1.17.0'

In [ ]:
import pathlib
FILES = {'__init__.py': '', 'steplength_env.py': '"""G1 step-length task: MuJoCo Playground\'s G1 joystick env + explicit step-length control.\n\nOperator command: forward speed vx and step length l*. Cadence follows from them:\n    gait_freq f = |vx| / (2 l*)      (two steps per gait cycle)\nso during training we sample (vx, f) over a wide range and give the policy both\nf and l* = vx / (2 f) as observations. A touchdown reward pays for landing the\nswing foot l* ahead of the stance foot along the heading.\n\nChanges vs. upstream G1 Joystick (kept deliberately small, see docs/pipeline.md):\n  * gait_freq range widened 1.25–1.5 Hz  ->  GAIT_FREQ_RANGE\n  * obs "state" gets [l*, f] appended (+2)\n  * new reward term "step_length"\n"""\nfrom __future__ import annotations\n\nimport jax\nimport jax.numpy as jp\nfrom ml_collections import config_dict\nfrom mujoco_playground._src import mjx_env\nfrom mujoco_playground._src.locomotion.g1 import joystick as g1_joystick\n\nmjx_env.ensure_menagerie_exists()  # downloads G1 meshes on first use\n\nGAIT_FREQ_RANGE = (0.9, 1.8)      # Hz  -> gait period 0.56–1.11 s\nSTEP_SIGMA = 0.05                 # m, width of touchdown reward\nMIN_WALK_SPEED = 0.15             # m/s, below this the step reward is off\n\n\ndef default_config() -> config_dict.ConfigDict:\n    cfg = g1_joystick.default_config()\n    cfg.reward_config.scales.step_length = 1.5\n    cfg.lin_vel_x = [-0.5, 1.2]\n    return cfg\n\n\nclass StepLength(g1_joystick.Joystick):\n\n    def __init__(self, task: str = "flat_terrain", config=None, config_overrides=None):\n        super().__init__(task=task, config=config or default_config(), config_overrides=config_overrides)\n\n    # -- command ---------------------------------------------------------------\n    def _sample_gait(self, rng, command):\n        f = jax.random.uniform(rng, (), minval=GAIT_FREQ_RANGE[0], maxval=GAIT_FREQ_RANGE[1])\n        step_cmd = command[0] / (2.0 * f)\n        return f, step_cmd\n\n    def _set_gait(self, info, rng):\n        f, step_cmd = self._sample_gait(rng, info["command"])\n        info["gait_freq"] = f\n        info["step_cmd"] = step_cmd\n        info["phase_dt"] = 2 * jp.pi * self.dt * jp.array([f])\n        return info\n\n    # -- env API ---------------------------------------------------------------\n    def reset(self, rng):\n        rng, gait_rng = jax.random.split(rng)\n        state = super().reset(rng)\n        info = self._set_gait(dict(state.info), gait_rng)\n        state.metrics["step_len_err"] = jp.zeros(())\n        contact = self._contact(state.data)\n        obs = self._get_obs(state.data, info, contact)\n        return state.replace(obs=obs, info=info)\n\n    def step(self, state, action):\n        state = super().step(state, action)\n        # Upstream resamples the velocity command when info["step"] wraps to 0; resample cadence too.\n        info = dict(state.info)\n        info["rng"], gait_rng = jax.random.split(info["rng"])\n        new = self._set_gait(dict(info), gait_rng)\n        wrapped = info["step"] == 0\n        for k in ("gait_freq", "step_cmd", "phase_dt"):\n            info[k] = jp.where(wrapped, new[k], info[k])\n        return state.replace(info=info)\n\n    def _contact(self, data):\n        return jp.array([\n            data.sensordata[self._mj_model.sensor_adr[s]] > 0 for s in self._feet_floor_found_sensor\n        ])\n\n    def _get_obs(self, data, info, contact):\n        obs = super()._get_obs(data, info, contact)\n        extra = jp.hstack([\n            info.get("step_cmd", jp.zeros(())) * 4.0,          # ~[-1, 1] scale\n            info.get("gait_freq", jp.array(1.35)) - 1.35,\n        ])\n        return {\n            "state": jp.hstack([obs["state"], extra]),\n            "privileged_state": jp.hstack([obs["privileged_state"], extra]),\n        }\n\n    def _get_reward(self, data, action, info, metrics, done, first_contact, contact):\n        rewards = super()._get_reward(data, action, info, metrics, done, first_contact, contact)\n        rewards["step_length"] = self._reward_step_length(data, info, first_contact, metrics)\n        return rewards\n\n    def _reward_step_length(self, data, info, first_contact, metrics):\n        feet_xy = data.site_xpos[self._feet_site_id][:, :2]\n        fwd = data.xmat[self._torso_body_id][:2, 0]\n        fwd = fwd / (jp.linalg.norm(fwd) + 1e-6)\n        # step length of foot i = how far it landed ahead of the other foot\n        step = jp.array([\n            jp.dot(feet_xy[0] - feet_xy[1], fwd),\n            jp.dot(feet_xy[1] - feet_xy[0], fwd),\n        ])\n        err = step - info.get("step_cmd", jp.zeros(()))\n        per_foot = jp.exp(-jp.square(err / STEP_SIGMA)) * first_contact\n        walking = jp.abs(info["command"][0]) > MIN_WALK_SPEED\n        # metric: abs error on touchdown steps (0 when no touchdown this tick)\n        n_td = jp.maximum(jp.sum(first_contact), 1.0)\n        metrics["step_len_err"] = jp.sum(jp.abs(err) * first_contact) / n_td * walking\n        return jp.sum(per_foot) * walking\n', 'train.py': '"""Train the G1 step-length policy with Brax PPO (MuJoCo Playground recipe).\n\nRuns anywhere JAX runs: Kaggle/Colab GPU for real training, the Mac CPU for a\nsmoke test.\n\n    python -m g1pipe.train --timesteps 150_000_000 --out runs/steplength_v1      # GPU\n    python -m g1pipe.train --smoke --out runs/smoke                               # CPU check\n\nWrites to --out:  params.pkl (final), ckpt_*.pkl (periodic), progress.csv, config.json\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport functools\nimport json\nimport pickle\nimport time\nfrom pathlib import Path\n\nimport jax\nfrom brax.training.agents.ppo import networks as ppo_networks\nfrom brax.training.agents.ppo import train as ppo\nfrom mujoco_playground import wrapper\nfrom mujoco_playground._src.locomotion.g1 import randomize as g1_randomize\nfrom mujoco_playground.config import locomotion_params\n\nfrom g1pipe.steplength_env import StepLength, default_config\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--out", default="runs/steplength")\n    ap.add_argument("--timesteps", type=int, default=150_000_000)\n    ap.add_argument("--num-envs", type=int, default=None)\n    ap.add_argument("--impl", default=None, help="\'warp\' (NVIDIA GPU) or \'jax\'; default picks by backend")\n    ap.add_argument("--no-dr", action="store_true", help="disable domain randomisation (experiment E3)")\n    ap.add_argument("--step-scale", type=float, default=None, help="override step_length reward scale")\n    ap.add_argument("--seed", type=int, default=0)\n    ap.add_argument("--smoke", action="store_true", help="tiny CPU run to check the pipeline end to end")\n    a = ap.parse_args()\n\n    out = Path(a.out)\n    out.mkdir(parents=True, exist_ok=True)\n    backend = jax.default_backend()\n    print("JAX backend:", backend, "devices:", jax.devices())\n\n    env_cfg = default_config()\n    env_cfg.impl = a.impl or ("warp" if backend == "gpu" else "jax")\n    if a.step_scale is not None:\n        env_cfg.reward_config.scales.step_length = a.step_scale\n    if a.no_dr:\n        env_cfg.push_config.enable = False\n        env_cfg.noise_config.level = 0.0\n\n    rl = locomotion_params.brax_ppo_config("G1JoystickFlatTerrain")\n    rl.num_timesteps = a.timesteps\n    if a.num_envs:\n        rl.num_envs = a.num_envs\n    if a.smoke:\n        rl.num_timesteps, rl.num_envs, rl.batch_size = 20_000, 32, 32\n        rl.num_minibatches, rl.num_evals, rl.episode_length = 4, 2, 100\n        rl.num_resets_per_eval = 0\n        env_cfg.naconmax, env_cfg.njmax = 64, env_cfg.njmax\n\n    env = StepLength(config=env_cfg)\n    eval_env = StepLength(config=env_cfg)\n\n    params_nf = dict(rl.network_factory)\n    train_kwargs = {k: v for k, v in rl.items() if k != "network_factory"}\n    network_factory = functools.partial(ppo_networks.make_ppo_networks, **params_nf)\n\n    (out / "config.json").write_text(json.dumps({\n        "env": env_cfg.to_dict(), "ppo": {**train_kwargs, "network_factory": params_nf},\n        "no_dr": a.no_dr, "seed": a.seed, "backend": backend,\n    }, indent=2, default=str))\n\n    t0 = time.time()\n    log = open(out / "progress.csv", "w", newline="")\n    writer = None\n\n    def progress(step, metrics):\n        nonlocal writer\n        row = {"step": step, "wall_s": round(time.time() - t0, 1),\n               **{k: float(v) for k, v in metrics.items() if k.startswith("eval/")}}\n        if writer is None:\n            writer = csv.DictWriter(log, fieldnames=list(row.keys()), extrasaction="ignore")\n            writer.writeheader()\n        writer.writerow(row)\n        log.flush()\n        print(f"[{row[\'wall_s\']:>7.0f}s] step {step:>11,}  reward {row.get(\'eval/episode_reward\', float(\'nan\')):8.2f}"\n              f"  step_len_err {row.get(\'eval/episode_step_len_err\', float(\'nan\')):.3f}", flush=True)\n\n    def save_ckpt(step, make_policy, params):\n        with open(out / f"ckpt_{step:011d}.pkl", "wb") as f:\n            pickle.dump(jax.device_get(params), f)\n\n    train_fn = functools.partial(\n        ppo.train, **train_kwargs, network_factory=network_factory, seed=a.seed,\n        randomization_fn=None if a.no_dr else g1_randomize.domain_randomize,\n        progress_fn=progress, policy_params_fn=save_ckpt,\n    )\n    make_inference_fn, params, _ = train_fn(\n        environment=env, eval_env=eval_env, wrap_env_fn=wrapper.wrap_for_brax_training)\n\n    with open(out / "params.pkl", "wb") as f:\n        pickle.dump({"params": jax.device_get(params), "network_factory": params_nf,\n                     "obs_size": env.observation_size, "action_size": env.action_size}, f)\n    print(f"done in {time.time() - t0:.0f}s -> {out}/params.pkl")\n\n\nif __name__ == "__main__":\n    main()\n'}
pkg = pathlib.Path('/content/src/g1pipe'); pkg.mkdir(parents=True, exist_ok=True)
for name, src in FILES.items():
    (pkg / name).write_text(src)
print('wrote', list(FILES))

In [ ]:
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = f'/content/drive/MyDrive/g1pipe_runs/{RUN_NAME}'
else:
    OUT = f'/content/runs/{RUN_NAME}'
os.makedirs(OUT, exist_ok=True); print(OUT)

In [ ]:
import subprocess, sys, os
env = dict(os.environ, PYTHONPATH='/content/src', XLA_PYTHON_CLIENT_MEM_FRACTION='0.9')
cmd = [sys.executable, '-u', '-m', 'g1pipe.train', '--out', OUT, '--timesteps', str(TIMESTEPS), *EXTRA_ARGS]
p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if 'Warning' not in line and 'warn(' not in line:
        print(line, end='')
print('exit code', p.wait())

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv(f'{OUT}/progress.csv'); display(df.tail())
df.plot(x='step', y=[c for c in df.columns if c in ('eval/episode_reward','eval/episode_reward/step_length')], subplots=True, figsize=(8,5)); plt.show()

In [ ]:
# Download params + progress (checkpoints stay on Drive)
import shutil, tempfile, pathlib
tmp = pathlib.Path(tempfile.mkdtemp()) / RUN_NAME / 'run'
shutil.copytree(OUT, tmp, ignore=shutil.ignore_patterns('ckpt_*'))
zip_path = shutil.make_archive(f'/content/{RUN_NAME}', 'zip', tmp.parent.parent)
from google.colab import files as _f; _f.download(zip_path)